# 客户流失预测项目 (优化版 V2)

> **课程**: SDSC8009 Data Mining and Knowledge Discovery  
> **任务**: 预测未来5个月客户流失风险  
> **数据**: ecommerce_customer_behavior_dataset_v2.csv (17,049条记录, 5,000客户)  
> **工具**: Polars + Scikit-learn + XGBoost + LightGBM + CatBoost + SMOTE

---

## 项目流程

1. 环境准备与数据加载
2. 数据清洗 (异常值处理)
3. 流失标签构建 (优化版)
4. 特征工程 (30个特征)
5. 合并特征和标签
6. 特征Winsorization
7. 数据分割与预处理
8. 特征选择 (SelectKBest, Top 20)
9. SMOTE处理类别不平衡
10. 模型训练 (6个模型)
11. 模型性能对比
12. 5-Fold交叉验证
13. 混淆矩阵分析
14. 特征重要性分析

---

## 核心优化 (V2)

- ✅ **优化1**: 调整时间窗口 (特征期9个月 vs 标签期5个月)
- ✅ **优化2**: 调整流失定义 (标签期内无购买 且 recency > 90天)
- ✅ **优化3**: 增加5个新特征 (周期性、趋势、集中度、敏感度、参与度)
- ✅ **优化4**: 添加CatBoost模型
- ✅ **优化5**: 特征选择优化 (SelectKBest, Top 20)

---

## 预期效果

- 🎯 **测试集AUC**: 0.75-0.85
- 🎯 **解决过拟合**: CV-Test差距 < 0.05
- 🎯 **提升稳定性**: CV标准差 < 0.05

---
## 阶段1: 环境准备与数据加载

导入所需的库，并加载电商客户行为数据集。

In [1]:
# 导入核心库
import polars as pl
import numpy as np
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("✅ 环境准备完成")

✅ 环境准备完成


In [2]:
# 加载数据
df = pl.read_csv('data/ecommerce_customer_behavior_dataset_v2.csv')

# 转换日期列
df = df.with_columns(
    pl.col('Date').str.to_date('%Y-%m-%d')
)

print("=" * 60)
print("数据加载完成")
print("=" * 60)
print(f"总记录数: {df.height:,}")
print(f"总客户数: {df['Customer_ID'].n_unique():,}")
print(f"时间范围: {df['Date'].min()} 至 {df['Date'].max()}")
print()

数据加载完成
总记录数: 17,049
总客户数: 5,000
时间范围: 2023-01-01 至 2024-03-25



---
## 阶段2: 数据清洗 (异常值处理)

使用99.5%分位数截尾法处理异常值，删除极端异常的订单记录。

In [3]:
# 计算99.5%分位数
total_amount_995 = df['Total_Amount'].quantile(0.995)
unit_price_995 = df['Unit_Price'].quantile(0.995)

print("=" * 60)
print("数据清洗 (异常值处理)")
print("=" * 60)
print(f"Total_Amount 99.5%分位数: {total_amount_995:.2f}")
print(f"Unit_Price 99.5%分位数: {unit_price_995:.2f}")
print()

# 删除异常值
df_clean = df.filter(
    (pl.col('Total_Amount') <= total_amount_995) &
    (pl.col('Unit_Price') <= unit_price_995)
)

print(f"删除前: {df.height:,}条记录")
print(f"删除后: {df_clean.height:,}条记录")
print(f"删除比例: {(df.height - df_clean.height) / df.height * 100:.2f}%")
print("✅ 数据清洗完成")
print()

数据清洗 (异常值处理)
Total_Amount 99.5%分位数: 15319.56
Unit_Price 99.5%分位数: 4537.80

删除前: 17,049条记录
删除后: 16,916条记录
删除比例: 0.78%
✅ 数据清洗完成



---
## 阶段3: 流失标签构建 (优化版)

**优化点**:
- **时间窗口调整**: 特征期9个月 (2023-01-01 至 2023-10-01) vs 标签期5个月 (2023-10-01 至 2024-03-01)
- **流失定义优化**: 标签期内无购买 **且** recency > 90天 = 流失

这样可以排除正常的低频购买客户，更精准地识别真正的流失客户。

In [4]:
# 定义时间窗口 (优化1)
cutoff_date = datetime(2023, 10, 1).date()  # 从2023-09-01改为2023-10-01
label_end_date = datetime(2024, 3, 1).date()  # 标签期5个月

print("=" * 60)
print("流失标签构建 (优化版)")
print("=" * 60)
print(f"Cutoff Date: {cutoff_date}")
print(f"特征期: 2023-01-01 至 {cutoff_date} (9个月)")
print(f"标签期: {cutoff_date} 至 {label_end_date} (5个月)")
print()

# 分割数据
df_feature = df_clean.filter(pl.col('Date') < cutoff_date)
df_label = df_clean.filter(
    (pl.col('Date') >= cutoff_date) & (pl.col('Date') < label_end_date)
)

# 获取客户列表
feature_customers = df_feature['Customer_ID'].unique().to_list()
label_customers = df_label['Customer_ID'].unique().to_list()

# 计算每个客户的最后购买日期
last_purchase = df_feature.group_by('Customer_ID').agg(
    pl.col('Date').max().alias('last_purchase_date')
)

# 构建流失标签 (优化2: 新的流失定义)
churn_threshold_days = 90
churn_customers = []

for customer_id in feature_customers:
    # 检查是否在标签期内有购买
    if customer_id not in label_customers:
        # 检查recency是否超过90天
        last_date = last_purchase.filter(pl.col('Customer_ID') == customer_id)['last_purchase_date'][0]
        recency_days = (cutoff_date - last_date).days
        if recency_days > churn_threshold_days:
            churn_customers.append(customer_id)

# 构建标签DataFrame
all_customers = pl.DataFrame({
    'Customer_ID': feature_customers,
    'is_churn': [1 if cid in churn_customers else 0 for cid in feature_customers]
})

print(f"标签统计:")
print(f"  - 总客户数: {len(feature_customers):,}")
print(f"  - 流失客户数: {len(churn_customers):,} ({len(churn_customers)/len(feature_customers)*100:.1f}%)")
print(f"  - 未流失客户数: {len(feature_customers) - len(churn_customers):,} ({(len(feature_customers) - len(churn_customers))/len(feature_customers)*100:.1f}%)")
print()

流失标签构建 (优化版)
Cutoff Date: 2023-10-01
特征期: 2023-01-01 至 2023-10-01 (9个月)
标签期: 2023-10-01 至 2024-03-01 (5个月)

标签统计:
  - 总客户数: 4,326
  - 流失客户数: 745 (17.2%)
  - 未流失客户数: 3,581 (82.8%)



---
## 阶段4: 特征工程 (30个特征)

**特征分类**:
- **RFM特征** (5个): recency, frequency, monetary, customer_lifetime_days, avg_days_between_orders
- **行为特征** (7个): n_categories, avg_rating, avg_delivery_days, avg_session, avg_pages, discount_rate, orders_last_30days
- **产品偏好特征** (4个): electronics_ratio, fashion_ratio, home_ratio, beauty_ratio
- **支付与设备特征** (3个): mobile_ratio, credit_card_ratio, is_returning_ratio
- **人口统计特征** (1个): age
- **创新特征** (5个): rfm_score, avg_order_value, purchase_intensity, monetary_trend, recency_ratio
- **新增特征** (5个): purchase_regularity, recent_3m_trend, category_concentration, price_sensitivity, engagement_score

**新增特征说明**:
1. **purchase_regularity**: 购买周期性 (标准差/均值，越小越规律)
2. **recent_3m_trend**: 最近3个月消费趋势 (相对于平均月消费的变化)
3. **category_concentration**: 品类集中度 (Herfindahl指数)
4. **price_sensitivity**: 价格敏感度 (折扣使用率变化)
5. **engagement_score**: 参与度得分 (session + pages + rating综合)

In [5]:
print("=" * 60)
print("特征工程 (30个特征)")
print("=" * 60)

# 初始化特征列表
customer_ids = []
# RFM特征 (5个)
recency_list = []
frequency_list = []
monetary_list = []
customer_lifetime_days_list = []
avg_days_between_orders_list = []

# 行为特征 (7个)
n_categories_list = []
avg_rating_list = []
avg_delivery_days_list = []
avg_session_list = []
avg_pages_list = []
discount_rate_list = []
orders_last_30days_list = []

# 产品偏好特征 (4个)
electronics_ratio_list = []
fashion_ratio_list = []
home_ratio_list = []
beauty_ratio_list = []

# 支付与设备特征 (3个)
mobile_ratio_list = []
credit_card_ratio_list = []
is_returning_ratio_list = []

# 人口统计特征 (1个)
age_list = []

# 创新特征 (5个)
rfm_score_list = []
avg_order_value_list = []
purchase_intensity_list = []
monetary_trend_list = []
recency_ratio_list = []

# 新增特征 (5个) - 优化3
purchase_regularity_list = []  # 购买周期性
recent_3m_trend_list = []  # 最近3个月消费趋势
category_concentration_list = []  # 品类集中度
price_sensitivity_list = []  # 价格敏感度
engagement_score_list = []  # 参与度得分

print("计算30个特征...")
print()

特征工程 (30个特征)
计算30个特征...



In [6]:
# 遍历每个客户计算特征
for customer_id in feature_customers:
    customer_data = df_feature.filter(pl.col('Customer_ID') == customer_id)
    
    # 基本信息
    customer_ids.append(customer_id)
    n_orders = customer_data.height
    
    # RFM特征
    last_purchase_date = customer_data['Date'].max()
    first_purchase_date = customer_data['Date'].min()
    recency = (cutoff_date - last_purchase_date).days
    frequency = n_orders
    monetary = customer_data['Total_Amount'].sum()
    customer_lifetime = (last_purchase_date - first_purchase_date).days + 1
    avg_days_between = customer_lifetime / max(frequency - 1, 1)
    
    recency_list.append(recency)
    frequency_list.append(frequency)
    monetary_list.append(monetary)
    customer_lifetime_days_list.append(customer_lifetime)
    avg_days_between_orders_list.append(avg_days_between)
    
    # 行为特征
    n_categories = customer_data['Product_Category'].n_unique()
    avg_rating = customer_data['Customer_Rating'].mean()
    avg_delivery = customer_data['Delivery_Time_Days'].mean()
    avg_session = customer_data['Session_Duration_Minutes'].mean()
    avg_pages = customer_data['Pages_Viewed'].mean()
    total_discount = customer_data['Discount_Amount'].sum()
    discount_rate = total_discount / monetary if monetary > 0 else 0
    
    # 最近30天订单数
    days_30_ago = cutoff_date - timedelta(days=30)
    orders_last_30 = customer_data.filter(pl.col('Date') >= days_30_ago).height
    
    n_categories_list.append(n_categories)
    avg_rating_list.append(avg_rating)
    avg_delivery_days_list.append(avg_delivery)
    avg_session_list.append(avg_session)
    avg_pages_list.append(avg_pages)
    discount_rate_list.append(discount_rate)
    orders_last_30days_list.append(orders_last_30)
    
    # 产品偏好特征
    category_counts = customer_data.group_by('Product_Category').agg(pl.len().alias('count'))
    total_orders = n_orders
    
    electronics_ratio = category_counts.filter(pl.col('Product_Category') == 'Electronics')['count'][0] / total_orders if len(category_counts.filter(pl.col('Product_Category') == 'Electronics')) > 0 else 0
    fashion_ratio = category_counts.filter(pl.col('Product_Category') == 'Fashion')['count'][0] / total_orders if len(category_counts.filter(pl.col('Product_Category') == 'Fashion')) > 0 else 0
    home_ratio = category_counts.filter(pl.col('Product_Category') == 'Home & Garden')['count'][0] / total_orders if len(category_counts.filter(pl.col('Product_Category') == 'Home & Garden')) > 0 else 0
    beauty_ratio = category_counts.filter(pl.col('Product_Category') == 'Beauty & Personal Care')['count'][0] / total_orders if len(category_counts.filter(pl.col('Product_Category') == 'Beauty & Personal Care')) > 0 else 0
    
    electronics_ratio_list.append(electronics_ratio)
    fashion_ratio_list.append(fashion_ratio)
    home_ratio_list.append(home_ratio)
    beauty_ratio_list.append(beauty_ratio)
    
    # 支付与设备特征
    mobile_count = customer_data.filter(pl.col('Device_Type') == 'Mobile')['Device_Type'].count()
    mobile_ratio = mobile_count / total_orders
    
    credit_card_count = customer_data.filter(pl.col('Payment_Method') == 'Credit Card')['Payment_Method'].count()
    credit_card_ratio = credit_card_count / total_orders
    
    is_returning_count = customer_data.filter(pl.col('Is_Returning_Customer') == 1)['Is_Returning_Customer'].count()
    is_returning_ratio = is_returning_count / total_orders
    
    mobile_ratio_list.append(mobile_ratio)
    credit_card_ratio_list.append(credit_card_ratio)
    is_returning_ratio_list.append(is_returning_ratio)
    
    # 人口统计特征
    age = customer_data['Age'][0]
    age_list.append(age)
    
    # 创新特征
    # RFM Score (简化版)
    r_score = 5 if recency < 30 else (4 if recency < 60 else (3 if recency < 90 else (2 if recency < 180 else 1)))
    f_score = 5 if frequency >= 8 else (4 if frequency >= 6 else (3 if frequency >= 4 else (2 if frequency >= 2 else 1)))
    m_score = 5 if monetary >= 5000 else (4 if monetary >= 3000 else (3 if monetary >= 1500 else (2 if monetary >= 500 else 1)))
    rfm_score = r_score * 100 + f_score * 10 + m_score
    
    avg_order_value = monetary / frequency
    purchase_intensity = frequency / (customer_lifetime / 30)  # 每月购买次数
    
    # 消费趋势 (前半期 vs 后半期)
    mid_date = first_purchase_date + timedelta(days=customer_lifetime // 2)
    first_half_monetary = customer_data.filter(pl.col('Date') < mid_date)['Total_Amount'].sum()
    second_half_monetary = customer_data.filter(pl.col('Date') >= mid_date)['Total_Amount'].sum()
    monetary_trend = (second_half_monetary - first_half_monetary) / max(first_half_monetary, 1)
    
    recency_ratio = recency / customer_lifetime if customer_lifetime > 0 else 0
    
    rfm_score_list.append(rfm_score)
    avg_order_value_list.append(avg_order_value)
    purchase_intensity_list.append(purchase_intensity)
    monetary_trend_list.append(monetary_trend)
    recency_ratio_list.append(recency_ratio)
    
    # 新增特征 (5个)
    # 1. 购买周期性 (标准差/均值，越小越规律)
    if frequency > 1:
        dates = sorted(customer_data['Date'].to_list())
        intervals = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
        if len(intervals) > 0:
            purchase_regularity = np.std(intervals) / (np.mean(intervals) + 1)
        else:
            purchase_regularity = 0
    else:
        purchase_regularity = 0
    purchase_regularity_list.append(purchase_regularity)
    
    # 2. 最近3个月消费趋势
    days_90_ago = cutoff_date - timedelta(days=90)
    recent_3m_data = customer_data.filter(pl.col('Date') >= days_90_ago)
    recent_3m_monetary = recent_3m_data['Total_Amount'].sum()
    avg_monthly_monetary = monetary / (customer_lifetime / 30)
    recent_3m_trend = (recent_3m_monetary / 3) / max(avg_monthly_monetary, 1) - 1
    recent_3m_trend_list.append(recent_3m_trend)
    
    # 3. 品类集中度 (Herfindahl指数)
    category_shares = []
    for cat in category_counts['Product_Category']:
        count = category_counts.filter(pl.col('Product_Category') == cat)['count'][0]
        share = count / total_orders
        category_shares.append(share ** 2)
    category_concentration = sum(category_shares)
    category_concentration_list.append(category_concentration)
    
    # 4. 价格敏感度 (折扣使用率变化)
    if frequency > 1:
        first_half_data = customer_data.filter(pl.col('Date') < mid_date)
        second_half_data = customer_data.filter(pl.col('Date') >= mid_date)
        first_half_discount_rate = first_half_data['Discount_Amount'].sum() / max(first_half_data['Total_Amount'].sum(), 1)
        second_half_discount_rate = second_half_data['Discount_Amount'].sum() / max(second_half_data['Total_Amount'].sum(), 1)
        price_sensitivity = second_half_discount_rate - first_half_discount_rate
    else:
        price_sensitivity = 0
    price_sensitivity_list.append(price_sensitivity)
    
    # 5. 参与度得分 (session + pages + rating综合)
    engagement_score = (avg_session / 30) * 0.3 + (avg_pages / 10) * 0.3 + (avg_rating / 5) * 0.4
    engagement_score_list.append(engagement_score)

print("✅ 特征计算完成")
print()

✅ 特征计算完成



In [7]:
# 构建特征DataFrame
features = pl.DataFrame({
    'Customer_ID': customer_ids,
    # RFM特征 (5个)
    'recency': recency_list,
    'frequency': frequency_list,
    'monetary': monetary_list,
    'customer_lifetime_days': customer_lifetime_days_list,
    'avg_days_between_orders': avg_days_between_orders_list,
    # 行为特征 (7个)
    'n_categories': n_categories_list,
    'avg_rating': avg_rating_list,
    'avg_delivery_days': avg_delivery_days_list,
    'avg_session': avg_session_list,
    'avg_pages': avg_pages_list,
    'discount_rate': discount_rate_list,
    'orders_last_30days': orders_last_30days_list,
    # 产品偏好特征 (4个)
    'electronics_ratio': electronics_ratio_list,
    'fashion_ratio': fashion_ratio_list,
    'home_ratio': home_ratio_list,
    'beauty_ratio': beauty_ratio_list,
    # 支付与设备特征 (3个)
    'mobile_ratio': mobile_ratio_list,
    'credit_card_ratio': credit_card_ratio_list,
    'is_returning_ratio': is_returning_ratio_list,
    # 人口统计特征 (1个)
    'age': age_list,
    # 创新特征 (5个)
    'rfm_score': rfm_score_list,
    'avg_order_value': avg_order_value_list,
    'purchase_intensity': purchase_intensity_list,
    'monetary_trend': monetary_trend_list,
    'recency_ratio': recency_ratio_list,
    # 新增特征 (5个)
    'purchase_regularity': purchase_regularity_list,
    'recent_3m_trend': recent_3m_trend_list,
    'category_concentration': category_concentration_list,
    'price_sensitivity': price_sensitivity_list,
    'engagement_score': engagement_score_list
}, strict=False)

print(f"特征工程完成:")
print(f"  - 特征数量: 30个 (不含Customer_ID)")
print(f"  - 样本数量: {features.height:,}个客户")
print()

特征工程完成:
  - 特征数量: 30个 (不含Customer_ID)
  - 样本数量: 4,326个客户



---
## 阶段5: 合并特征和标签

将特征数据和流失标签合并，形成完整的建模数据集。

In [8]:
# 合并特征和标签
df_final = features.join(all_customers, on='Customer_ID', how='inner')

print("=" * 60)
print("合并特征和标签")
print("=" * 60)
print(f"合并后样本数: {df_final.height:,}")
print(f"特征数: 30")
print(f"标签分布:")
churn_count = df_final.filter(pl.col('is_churn') == 1).height
non_churn_count = df_final.filter(pl.col('is_churn') == 0).height
print(f"  - 流失 (1): {churn_count:,} ({churn_count/df_final.height*100:.1f}%)")
print(f"  - 未流失 (0): {non_churn_count:,} ({non_churn_count/df_final.height*100:.1f}%)")
print()

合并特征和标签
合并后样本数: 4,326
特征数: 30
标签分布:
  - 流失 (1): 745 (17.2%)
  - 未流失 (0): 3,581 (82.8%)



---
## 阶段6: 特征Winsorization (99%)

使用99%分位数进行Winsorization处理，将极端值限制在合理范围内，减少异常值对模型的影响。

In [9]:
print("=" * 60)
print("特征Winsorization (99%)")
print("=" * 60)

feature_cols = [col for col in df_final.columns if col not in ['Customer_ID', 'is_churn']]

for col in feature_cols:
    lower = df_final[col].quantile(0.01)
    upper = df_final[col].quantile(0.99)
    df_final = df_final.with_columns(
        pl.col(col).clip(lower, upper)
    )

print("✅ Winsorization完成")
print()

特征Winsorization (99%)
✅ Winsorization完成



---
## 阶段7: 数据分割与预处理

将数据分割为训练集和测试集 (80/20)，并使用RobustScaler进行标准化处理。

In [10]:
print("=" * 60)
print("数据分割与预处理")
print("=" * 60)

X = df_final.select(feature_cols).to_numpy()
y = df_final['is_churn'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"训练集: {X_train.shape[0]:,}个样本")
print(f"  - 流失: {np.sum(y_train == 1):,} ({np.sum(y_train == 1)/len(y_train)*100:.1f}%)")
print(f"  - 未流失: {np.sum(y_train == 0):,} ({np.sum(y_train == 0)/len(y_train)*100:.1f}%)")
print(f"测试集: {X_test.shape[0]:,}个样本")
print(f"  - 流失: {np.sum(y_test == 1):,} ({np.sum(y_test == 1)/len(y_test)*100:.1f}%)")
print(f"  - 未流失: {np.sum(y_test == 0):,} ({np.sum(y_test == 0)/len(y_test)*100:.1f}%)")
print()

# 标准化
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ 数据分割与标准化完成")
print()

数据分割与预处理
训练集: 3,460个样本
  - 流失: 596 (17.2%)
  - 未流失: 2,864 (82.8%)
测试集: 866个样本
  - 流失: 149 (17.2%)
  - 未流失: 717 (82.8%)

✅ 数据分割与标准化完成



---
## 阶段8: 特征选择 (优化5)

使用SelectKBest和mutual_info_classif选择Top 20特征，降低特征维度，减少过拟合风险。

In [11]:
print("=" * 60)
print("特征选择 (SelectKBest, Top 20)")
print("=" * 60)

# 使用mutual_info_classif选择Top 20特征
selector = SelectKBest(score_func=mutual_info_classif, k=20)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

# 获取选中的特征名
selected_features_mask = selector.get_support()
selected_features = [feature_cols[i] for i, selected in enumerate(selected_features_mask) if selected]

print(f"原始特征数: 30")
print(f"选择特征数: 20")
print(f"选中的特征:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i}. {feat}")
print()

特征选择 (SelectKBest, Top 20)
原始特征数: 30
选择特征数: 20
选中的特征:
  1. recency
  2. frequency
  3. monetary
  4. customer_lifetime_days
  5. avg_days_between_orders
  6. n_categories
  7. avg_delivery_days
  8. avg_session
  9. avg_pages
  10. orders_last_30days
  11. home_ratio
  12. mobile_ratio
  13. credit_card_ratio
  14. rfm_score
  15. purchase_intensity
  16. recency_ratio
  17. purchase_regularity
  18. recent_3m_trend
  19. category_concentration
  20. engagement_score



---
## 阶段9: SMOTE处理类别不平衡

使用SMOTE (Synthetic Minority Over-sampling Technique) 对训练集进行过采样，平衡流失和未流失客户的比例。

In [12]:
print("=" * 60)
print("SMOTE处理类别不平衡")
print("=" * 60)

print(f"SMOTE前训练集: {X_train_selected.shape[0]:,}个样本")
print(f"  - 流失: {np.sum(y_train == 1):,} ({np.sum(y_train == 1)/len(y_train)*100:.1f}%)")
print(f"  - 未流失: {np.sum(y_train == 0):,} ({np.sum(y_train == 0)/len(y_train)*100:.1f}%)")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_selected, y_train)

print(f"SMOTE后训练集: {X_train_resampled.shape[0]:,}个样本")
print(f"  - 流失: {np.sum(y_train_resampled == 1):,} ({np.sum(y_train_resampled == 1)/len(y_train_resampled)*100:.1f}%)")
print(f"  - 未流失: {np.sum(y_train_resampled == 0):,} ({np.sum(y_train_resampled == 0)/len(y_train_resampled)*100:.1f}%)")
print("✅ SMOTE完成")
print()

SMOTE处理类别不平衡
SMOTE前训练集: 3,460个样本
  - 流失: 596 (17.2%)
  - 未流失: 2,864 (82.8%)


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "d:\softwares\Anaconda3\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "d:\softwares\Anaconda3\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "d:\softwares\Anaconda3\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\softwares\Anaconda3\Lib\subprocess.py", line 1615, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "<frozen codecs>", line 325, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xce in position 4: invalid continuation byte


SMOTE后训练集: 5,728个样本
  - 流失: 2,864 (50.0%)
  - 未流失: 2,864 (50.0%)
✅ SMOTE完成



---
## 阶段10: 模型训练 (6个模型)

训练6个分类模型:
1. **Logistic Regression** - 基线模型
2. **Random Forest** - 集成学习
3. **XGBoost** - 梯度提升 (带GridSearchCV超参数优化)
4. **LightGBM** - 轻量级梯度提升
5. **CatBoost** - 类别特征友好的梯度提升 (新增)
6. **Voting Classifier** - 软投票集成

### 10.1 Logistic Regression

In [13]:
print("=" * 60)
print("10.1 Logistic Regression")
print("=" * 60)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_resampled, y_train_resampled)
y_pred_lr = lr.predict(X_test_selected)
y_pred_proba_lr = lr.predict_proba(X_test_selected)[:, 1]

auc_lr = roc_auc_score(y_test, y_pred_proba_lr)
acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print(f"测试集 - AUC: {auc_lr:.4f}, Accuracy: {acc_lr:.4f}, Precision: {prec_lr:.4f}, Recall: {rec_lr:.4f}, F1: {f1_lr:.4f}")
print()

10.1 Logistic Regression
测试集 - AUC: 0.8736, Accuracy: 0.7517, Precision: 0.4062, Recall: 0.9597, F1: 0.5709



### 10.2 Random Forest

In [14]:
print("=" * 60)
print("10.2 Random Forest")
print("=" * 60)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_resampled, y_train_resampled)
y_pred_rf = rf.predict(X_test_selected)
y_pred_proba_rf = rf.predict_proba(X_test_selected)[:, 1]

auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print(f"测试集 - AUC: {auc_rf:.4f}, Accuracy: {acc_rf:.4f}, Precision: {prec_rf:.4f}, Recall: {rec_rf:.4f}, F1: {f1_rf:.4f}")
print()

10.2 Random Forest
测试集 - AUC: 0.8705, Accuracy: 0.7633, Precision: 0.4108, Recall: 0.8658, F1: 0.5572



### 10.3 XGBoost (带GridSearchCV)

In [15]:
print("=" * 60)
print("10.3 XGBoost (带GridSearchCV)")
print("=" * 60)

print("开始超参数搜索 (3-Fold CV)...")
xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [6, 8],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb = XGBClassifier(random_state=42, eval_metric='logloss')
grid_search = GridSearchCV(xgb, xgb_params, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0)
grid_search.fit(X_train_resampled, y_train_resampled)

print(f"最佳参数: {grid_search.best_params_}")
print(f"最佳CV得分 (AUC): {grid_search.best_score_:.4f}")
print()

best_xgb = grid_search.best_estimator_
y_pred_xgb = best_xgb.predict(X_test_selected)
y_pred_proba_xgb = best_xgb.predict_proba(X_test_selected)[:, 1]

auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

print(f"测试集 - AUC: {auc_xgb:.4f}, Accuracy: {acc_xgb:.4f}, Precision: {prec_xgb:.4f}, Recall: {rec_xgb:.4f}, F1: {f1_xgb:.4f}")
print()

10.3 XGBoost (带GridSearchCV)
开始超参数搜索 (3-Fold CV)...
最佳参数: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 200, 'subsample': 1.0}
最佳CV得分 (AUC): 0.9725

测试集 - AUC: 0.8551, Accuracy: 0.8141, Precision: 0.4531, Recall: 0.3893, F1: 0.4188



### 10.4 LightGBM

In [16]:
print("=" * 60)
print("10.4 LightGBM")
print("=" * 60)

lgbm = LGBMClassifier(n_estimators=100, max_depth=10, random_state=42, verbose=-1)
lgbm.fit(X_train_resampled, y_train_resampled)
y_pred_lgbm = lgbm.predict(X_test_selected)
y_pred_proba_lgbm = lgbm.predict_proba(X_test_selected)[:, 1]

auc_lgbm = roc_auc_score(y_test, y_pred_proba_lgbm)
acc_lgbm = accuracy_score(y_test, y_pred_lgbm)
prec_lgbm = precision_score(y_test, y_pred_lgbm)
rec_lgbm = recall_score(y_test, y_pred_lgbm)
f1_lgbm = f1_score(y_test, y_pred_lgbm)

print(f"测试集 - AUC: {auc_lgbm:.4f}, Accuracy: {acc_lgbm:.4f}, Precision: {prec_lgbm:.4f}, Recall: {rec_lgbm:.4f}, F1: {f1_lgbm:.4f}")
print()

10.4 LightGBM
测试集 - AUC: 0.8609, Accuracy: 0.8164, Precision: 0.4675, Recall: 0.4832, F1: 0.4752



### 10.5 CatBoost (新增)

In [17]:
print("=" * 60)
print("10.5 CatBoost (新增)")
print("=" * 60)

catboost = CatBoostClassifier(
    iterations=100,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=0
)
catboost.fit(X_train_resampled, y_train_resampled)
y_pred_catboost = catboost.predict(X_test_selected)
y_pred_proba_catboost = catboost.predict_proba(X_test_selected)[:, 1]

auc_catboost = roc_auc_score(y_test, y_pred_proba_catboost)
acc_catboost = accuracy_score(y_test, y_pred_catboost)
prec_catboost = precision_score(y_test, y_pred_catboost)
rec_catboost = recall_score(y_test, y_pred_catboost)
f1_catboost = f1_score(y_test, y_pred_catboost)

print(f"测试集 - AUC: {auc_catboost:.4f}, Accuracy: {acc_catboost:.4f}, Precision: {prec_catboost:.4f}, Recall: {rec_catboost:.4f}, F1: {f1_catboost:.4f}")
print()

10.5 CatBoost (新增)
测试集 - AUC: 0.8735, Accuracy: 0.8083, Precision: 0.4612, Recall: 0.6779, F1: 0.5489



### 10.6 Voting Classifier (集成)

In [18]:
print("=" * 60)
print("10.6 Voting Classifier (集成)")
print("=" * 60)

print("训练Voting Classifier...")
voting = VotingClassifier(
    estimators=[
        ('lr', lr),
        ('rf', rf),
        ('xgb', best_xgb),
        ('lgbm', lgbm),
        ('catboost', catboost)
    ],
    voting='soft'
)
voting.fit(X_train_resampled, y_train_resampled)
y_pred_voting = voting.predict(X_test_selected)
y_pred_proba_voting = voting.predict_proba(X_test_selected)[:, 1]

auc_voting = roc_auc_score(y_test, y_pred_proba_voting)
acc_voting = accuracy_score(y_test, y_pred_voting)
prec_voting = precision_score(y_test, y_pred_voting)
rec_voting = recall_score(y_test, y_pred_voting)
f1_voting = f1_score(y_test, y_pred_voting)

print(f"测试集 - AUC: {auc_voting:.4f}, Accuracy: {acc_voting:.4f}, Precision: {prec_voting:.4f}, Recall: {rec_voting:.4f}, F1: {f1_voting:.4f}")
print()

10.6 Voting Classifier (集成)
训练Voting Classifier...
测试集 - AUC: 0.8687, Accuracy: 0.7875, Precision: 0.4229, Recall: 0.6443, F1: 0.5106



---
## 阶段11: 模型性能对比

汇总所有模型的性能指标，找出最佳模型。

In [19]:
print("=" * 60)
print("模型性能对比")
print("=" * 60)

# 汇总结果
results = [
    {'Model': 'Logistic Regression', 'AUC': auc_lr, 'Accuracy': acc_lr, 'Precision': prec_lr, 'Recall': rec_lr, 'F1-Score': f1_lr},
    {'Model': 'Random Forest', 'AUC': auc_rf, 'Accuracy': acc_rf, 'Precision': prec_rf, 'Recall': rec_rf, 'F1-Score': f1_rf},
    {'Model': 'XGBoost', 'AUC': auc_xgb, 'Accuracy': acc_xgb, 'Precision': prec_xgb, 'Recall': rec_xgb, 'F1-Score': f1_xgb},
    {'Model': 'LightGBM', 'AUC': auc_lgbm, 'Accuracy': acc_lgbm, 'Precision': prec_lgbm, 'Recall': rec_lgbm, 'F1-Score': f1_lgbm},
    {'Model': 'CatBoost', 'AUC': auc_catboost, 'Accuracy': acc_catboost, 'Precision': prec_catboost, 'Recall': rec_catboost, 'F1-Score': f1_catboost},
    {'Model': 'Voting Classifier', 'AUC': auc_voting, 'Accuracy': acc_voting, 'Precision': prec_voting, 'Recall': rec_voting, 'F1-Score': f1_voting}
]

results_df = pl.DataFrame(results)
print(results_df)
print()

# 找出最佳模型
best_model_name = max(results, key=lambda x: x['AUC'])['Model']
best_auc = max(results, key=lambda x: x['AUC'])['AUC']
print(f"🏆 最佳模型: {best_model_name} (AUC={best_auc:.4f})")
print()

# 保存最佳模型引用
models = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': best_xgb,
    'LightGBM': lgbm,
    'CatBoost': catboost,
    'Voting Classifier': voting
}
best_model = models[best_model_name]

模型性能对比
shape: (6, 6)
┌─────────────────────┬──────────┬──────────┬───────────┬──────────┬──────────┐
│ Model               ┆ AUC      ┆ Accuracy ┆ Precision ┆ Recall   ┆ F1-Score │
│ ---                 ┆ ---      ┆ ---      ┆ ---       ┆ ---      ┆ ---      │
│ str                 ┆ f64      ┆ f64      ┆ f64       ┆ f64      ┆ f64      │
╞═════════════════════╪══════════╪══════════╪═══════════╪══════════╪══════════╡
│ Logistic Regression ┆ 0.873635 ┆ 0.751732 ┆ 0.40625   ┆ 0.959732 ┆ 0.570858 │
│ Random Forest       ┆ 0.870461 ┆ 0.763279 ┆ 0.410828  ┆ 0.865772 ┆ 0.557235 │
│ XGBoost             ┆ 0.855101 ┆ 0.814088 ┆ 0.453125  ┆ 0.389262 ┆ 0.418773 │
│ LightGBM            ┆ 0.860867 ┆ 0.816397 ┆ 0.467532  ┆ 0.483221 ┆ 0.475248 │
│ CatBoost            ┆ 0.873504 ┆ 0.808314 ┆ 0.461187  ┆ 0.677852 ┆ 0.548913 │
│ Voting Classifier   ┆ 0.868739 ┆ 0.787529 ┆ 0.422907  ┆ 0.644295 ┆ 0.510638 │
└─────────────────────┴──────────┴──────────┴───────────┴──────────┴──────────┘

🏆 最佳模型: Logistic R

---
## 阶段12: 5-Fold交叉验证 (最佳模型)

对最佳模型进行5折交叉验证，评估模型的稳定性和泛化能力。

In [20]:
print("=" * 60)
print("5-Fold交叉验证 (最佳模型)")
print("=" * 60)

cv_scores = cross_val_score(best_model, X_train_resampled, y_train_resampled, cv=5, scoring='roc_auc', n_jobs=-1)

print(f"5-Fold CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"各Fold AUC: {cv_scores}")
print()

5-Fold交叉验证 (最佳模型)
5-Fold CV AUC: 0.8448 (+/- 0.0161)
各Fold AUC: [0.84415936 0.83079168 0.87584709 0.8360091  0.83720817]



---
## 阶段13: 混淆矩阵分析 (最佳模型)

分析最佳模型的混淆矩阵和分类报告，了解模型在不同类别上的表现。

In [21]:
print("=" * 60)
print("混淆矩阵 (最佳模型)")
print("=" * 60)

y_pred_best = best_model.predict(X_test_selected)
cm = confusion_matrix(y_test, y_pred_best)

print("混淆矩阵:")
print(f"              预测未流失  预测流失")
print(f"实际未流失       {cm[0][0]}          {cm[0][1]}")
print(f"实际流失         {cm[1][0]}          {cm[1][1]}")
print()

print("分类报告:")
print(classification_report(y_test, y_pred_best, target_names=['未流失', '流失']))
print()

混淆矩阵 (最佳模型)
混淆矩阵:
              预测未流失  预测流失
实际未流失       508          209
实际流失         6          143

分类报告:
              precision    recall  f1-score   support

         未流失       0.99      0.71      0.83       717
          流失       0.41      0.96      0.57       149

    accuracy                           0.75       866
   macro avg       0.70      0.83      0.70       866
weighted avg       0.89      0.75      0.78       866




---
## 阶段14: 特征重要性分析

分析XGBoost模型的特征重要性，识别对流失预测最重要的特征。

In [22]:
print("=" * 60)
print("特征重要性分析 (XGBoost)")
print("=" * 60)

feature_importance = best_xgb.feature_importances_
importance_df = pl.DataFrame({
    'Feature': selected_features,
    'Importance': feature_importance
}).sort('Importance', descending=True)

print("Top 10重要特征:")
print(importance_df.head(10))
print()

特征重要性分析 (XGBoost)
Top 10重要特征:
shape: (10, 2)
┌────────────────────────┬────────────┐
│ Feature                ┆ Importance │
│ ---                    ┆ ---        │
│ str                    ┆ f32        │
╞════════════════════════╪════════════╡
│ recency                ┆ 0.19565    │
│ orders_last_30days     ┆ 0.174578   │
│ recent_3m_trend        ┆ 0.129889   │
│ rfm_score              ┆ 0.105802   │
│ home_ratio             ┆ 0.039073   │
│ frequency              ┆ 0.032108   │
│ avg_delivery_days      ┆ 0.031311   │
│ avg_pages              ┆ 0.028936   │
│ n_categories           ┆ 0.028836   │
│ category_concentration ┆ 0.028402   │
└────────────────────────┴────────────┘



---
## 项目总结

汇总项目的核心成果、最终效果和业务洞察。

In [23]:
print("=" * 60)
print("项目总结")
print("=" * 60)
print()

print("✅ 核心成果:")
print("  1. 优化时间窗口: 特征期9个月 vs 标签期5个月")
print("  2. 优化流失定义: 标签期内无购买 且 recency > 90天")
print("  3. 特征数量: 30个 → 20个 (SelectKBest)")
print("  4. 数据处理: 99.5%截尾 + 99% Winsorization + SMOTE")
print("  5. 模型: LR + RF + XGBoost + LightGBM + CatBoost + Voting (6个)")
print("  6. 超参数优化: GridSearchCV (3-Fold CV)")
print("  7. 集成方法: Voting Classifier (Soft Voting)")
print()

print("✅ 最终效果:")
print(f"  - 最佳模型: {best_model_name}")
print(f"  - 测试集AUC: {best_auc:.4f}")
print(f"  - 5-Fold CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print()

if best_auc >= 0.75:
    print("🎉 达到预期目标 (AUC >= 0.75)!")
elif best_auc >= 0.70:
    print("⚠️ 接近预期目标 (AUC >= 0.70)")
else:
    print("⚠️ 未达到预期目标 (AUC < 0.70)")
print()

print("业务洞察:")
print(f"  - Top 3特征: {', '.join(importance_df['Feature'].head(3).to_list())}")
print(f"  - 流失率: {churn_count/df_final.height*100:.1f}%")
print(f"  - 模型可以识别 {rec_voting*100:.1f}% 的流失客户")
print(f"  - 预测为流失的客户中，{prec_voting*100:.1f}% 确实会流失")
print()

print("下一步建议:")
print("  1. 基于流失预测结果，制定客户挽回策略")
print("  2. 针对高风险客户，提供个性化优惠")
print("  3. 分析流失客户的共同特征，优化产品和服务")
print("  4. 定期重新训练模型，保持预测准确性")
print()

项目总结

✅ 核心成果:
  1. 优化时间窗口: 特征期9个月 vs 标签期5个月
  2. 优化流失定义: 标签期内无购买 且 recency > 90天
  3. 特征数量: 30个 → 20个 (SelectKBest)
  4. 数据处理: 99.5%截尾 + 99% Winsorization + SMOTE
  5. 模型: LR + RF + XGBoost + LightGBM + CatBoost + Voting (6个)
  6. 超参数优化: GridSearchCV (3-Fold CV)
  7. 集成方法: Voting Classifier (Soft Voting)

✅ 最终效果:
  - 最佳模型: Logistic Regression
  - 测试集AUC: 0.8736
  - 5-Fold CV AUC: 0.8448 (+/- 0.0161)

🎉 达到预期目标 (AUC >= 0.75)!

业务洞察:
  - Top 3特征: recency, orders_last_30days, recent_3m_trend
  - 流失率: 17.2%
  - 模型可以识别 64.4% 的流失客户
  - 预测为流失的客户中，42.3% 确实会流失

下一步建议:
  1. 基于流失预测结果，制定客户挽回策略
  2. 针对高风险客户，提供个性化优惠
  3. 分析流失客户的共同特征，优化产品和服务
  4. 定期重新训练模型，保持预测准确性

